# Cars.com Crawler — Colab Driver

All scraping logic lives in the `crawler/` Python package at the repo root.
This notebook is a **thin orchestration layer** for Colab. For local runs, prefer
the CLI: `python -m crawler discover --from 1 --to 10`.

**Architecture:** httpx (HTTP/2) for detail/seller/review pages, Selenium driver pool
(reused, not per-URL) for results pagination. Tenacity retries with exponential
backoff and 429-aware. Stable SHA1 cache keys so reruns resume cleanly.

**Selenium 4.43.0** ships with Selenium Manager — Chrome and ChromeDriver are
discovered and (if missing) downloaded automatically, so there is no longer a
manual `apt-get install google-chrome` step or `webdriver-manager` dependency.

**Output schema is unchanged** from the legacy notebook — `transform_raw_data.ipynb`
keeps working without modification.

## 1. Mount Drive (Colab only)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install crawler dependencies

Pinned in `crawler/requirements.txt`: `selenium 4.43.0` (with built-in Selenium
Manager), `httpx[http2]`, `lxml`, `tenacity`, `beautifulsoup4`. The first call
to `webdriver.Chrome()` will download Chrome-for-Testing into Selenium
Manager's cache (~150 MB) — that's it, nothing else to install at the OS level.

In [1]:
# When running on Colab from a Drive checkout, %cd to the repo root first.
# %cd /content/drive/MyDrive/car-recsys-consultant-chatbot
!pip install -q -r crawler/requirements.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
apache-airflow 2.7.2 requires sqlalchemy<2.0,>=1.4.28, but you have sqlalchemy 2.0.45 which is incompatible.
connexion 3.2.0 requires python-multipart>=0.0.15, but you have python-multipart 0.0.6 which is incompatible.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 3. Sanity-check the WebDriver

First run is slower (Selenium Manager downloads Chrome-for-Testing); subsequent
runs reuse the cached binaries.

In [2]:
from crawler.driver import build_driver
from crawler.settings import CrawlerSettings
from crawler.utils import configure_logging

configure_logging()
_d = build_driver(CrawlerSettings())
_d.get('https://www.google.com')
print('Title:', _d.title)
_d.quit()

Title: Google


## 4. Discover listing URLs

Writes one `car_links/page_<n>.txt` per page. Already-discovered pages are
skipped when `resume=True` (default).

In [4]:
from pathlib import Path
from crawler import discover_listing_urls, CrawlerSettings

settings = CrawlerSettings(
    link_dir=Path('car_links'),
    start_page=1,
    end_page=2,
)
discover_listing_urls(settings)

01:03:13 [INFO] crawler.pipeline: Discovering page 1
01:03:33 [WARNING] crawler.pipeline: Page 1 navigation failed: Message: 
Stacktrace:
#0 0x59a0cca68a6a <unknown>
#1 0x59a0cc477ab5 <unknown>
#2 0x59a0cc4ca676 <unknown>
#3 0x59a0cc4ca8b1 <unknown>
#4 0x59a0cc515614 <unknown>
#5 0x59a0cc5127b6 <unknown>
#6 0x59a0cc4bdcbf <unknown>
#7 0x59a0cc4bea81 <unknown>
#8 0x59a0cca2ea64 <unknown>
#9 0x59a0cca31951 <unknown>
#10 0x59a0cca1b21e <unknown>
#11 0x59a0cca3251e <unknown>
#12 0x59a0cca01be0 <unknown>
#13 0x59a0cca559b8 <unknown>
#14 0x59a0cca55b88 <unknown>
#15 0x59a0cca674de <unknown>
#16 0x762efec9caa4 <unknown>
#17 0x762efed29c6c <unknown>

01:03:33 [INFO] crawler.pipeline: Discovering page 2
01:03:54 [WARNING] crawler.pipeline: Page 2 navigation failed: Message: 
Stacktrace:
#0 0x59a0cca68a6a <unknown>
#1 0x59a0cc477ab5 <unknown>
#2 0x59a0cc4ca676 <unknown>
#3 0x59a0cc4ca8b1 <unknown>
#4 0x59a0cc515614 <unknown>
#5 0x59a0cc5127b6 <unknown>
#6 0x59a0cc4bdcbf <unknown>
#7 0x59a0cc4bea

{}

## 5. Scrape listings, sellers, and reviews

Tunables:

- `http_workers` — parallel httpx requests (default 16, safe for cars.com).
- `selenium_workers` — fallback driver pool size (default 2; only used when httpx is blocked).
- `parser_workers` — JSON-merge thread pool.
- `resume=True` — skip cars whose JSON already exists; on by default.

In [ ]:
from pathlib import Path
from crawler import scrape_listings, CrawlerSettings

settings = CrawlerSettings(
    link_dir=Path('car_links'),
    output_dir=Path('raw_data'),
    html_cache_dir=Path('html_cache'),
    start_page=170,
    end_page=200,
    http_workers=16,
    selenium_workers=2,
    parser_workers=8,
)
scrape_listings(settings)

## 6. Sync results to Drive

In [ ]:
!mkdir -p "/content/drive/MyDrive/Car Recsys Consultant Chatbot/"
!cp -r raw_data/* "/content/drive/MyDrive/Car Recsys Consultant Chatbot/"